# CASMI 2026 — Phase 1 Baseline: Library Search

**Method:** spectral entropy similarity vs 2.5M training spectra, RRF(k=60) fusion
across spectra per molecule, InChIKey14 dedupe, top-25 output.

**Pipeline:**
1. Load train/test parquet
2. Preprocess all train spectra (peak filter, sqrt scaling, L2 norm)
3. Build neutral-mass index (electron-mass-corrected adduct table)
4. **Validation:** hold out `enveda-np-examples` spectra, measure MRR@25
5. **Prediction:** per test molecule → ranked InChIKey14 list → submission.csv


In [ ]:

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import os, time, glob
from collections import defaultdict

print("numpy", np.__version__)
print("pandas", pd.__version__)


In [ ]:

# ================= Core library (Phase 1) =================
ELECTRON = 0.00054858
PROTON = 1.00782503223 - ELECTRON
NA_ION = 22.9897692809 - ELECTRON
K_ION = 38.9637074864 - ELECTRON
NH4_ION = 18.033823
H2O = 18.010564683
CL_ION = 34.968852682 + ELECTRON
FORMATE_ADDUCT = 46.005480

ADDUCT_DELTA = {
    "[M+H]+": -PROTON,
    "[M+NH4]+": -NH4_ION,
    "[M-H2O+H]+": -(PROTON - H2O),
    "[M-2H2O+H]+": -(PROTON - 2 * H2O),
    "[M+Na]+": -NA_ION,
    "[M+K]+": -K_ION,
    "[M-H]-": PROTON,
    "[M-H2O-H]-": PROTON + H2O,
    "[M+CH2O2-H]-": -(FORMATE_ADDUCT - PROTON),
    "[M+Cl]-": -CL_ION,
}

def neutral_mass(precursor_mz, adduct):
    precursor_mz = np.asarray(precursor_mz, dtype=np.float64)
    if isinstance(adduct, str):
        return precursor_mz + ADDUCT_DELTA[adduct]
    deltas = np.array([ADDUCT_DELTA.get(a, np.nan) for a in adduct])
    # unknown adducts -> fall back to delta 0 with a warning (never crash)
    bad = np.isnan(deltas)
    if bad.any():
        print(f"WARNING: unknown adducts {set(np.asarray(adduct)[bad])}, using delta=0")
        deltas[bad] = 0.0
    return precursor_mz + deltas

def preprocess_peaks(mzs, intensities, precursor_mz,
                     min_rel_int=0.01, max_mz_over_precursor=2.0):
    mzs = np.asarray(mzs, dtype=np.float64)
    intensities = np.asarray(intensities, dtype=np.float64)
    if len(mzs) == 0:
        return mzs, intensities
    order = np.argsort(mzs)
    mzs, intensities = mzs[order], intensities[order]
    base = intensities.max()
    if base <= 0:
        return np.empty(0), np.empty(0)
    keep = (intensities >= min_rel_int * base) & (mzs <= precursor_mz + max_mz_over_precursor)
    mzs, intensities = mzs[keep], intensities[keep]
    if len(mzs) == 0:
        return mzs, intensities
    intensities = np.sqrt(intensities)
    intensities /= np.linalg.norm(intensities)
    return mzs, intensities

def entropy_similarity(mzs_a, int_a, mzs_b, int_b, fragment_tol=0.01):
    mzs_a = np.asarray(mzs_a, dtype=np.float64); int_a = np.asarray(int_a, dtype=np.float64)
    mzs_b = np.asarray(mzs_b, dtype=np.float64); int_b = np.asarray(int_b, dtype=np.float64)
    if len(mzs_a) == 0 or len(mzs_b) == 0:
        return 0.0
    pa = int_a / int_a.sum(); pb = int_b / int_b.sum()
    s_a = float(-np.sum(pa * np.log(pa + 1e-300)))
    s_b = float(-np.sum(pb * np.log(pb + 1e-300)))
    i = j = 0; na, nb = len(mzs_a), len(mzs_b); merged = []
    while i < na and j < nb:
        dm = mzs_a[i] - mzs_b[j]
        if abs(dm) <= fragment_tol:
            merged.append(pa[i] + pb[j]); i += 1; j += 1
        elif dm < 0:
            merged.append(pa[i]); i += 1
        else:
            merged.append(pb[j]); j += 1
    while i < na: merged.append(pa[i]); i += 1
    while j < nb: merged.append(pb[j]); j += 1
    merged = np.array(merged); merged /= merged.sum()
    s_ab = float(-np.sum(merged * np.log(merged + 1e-300)))
    sim = 1.0 - (2.0 * s_ab - s_a - s_b) / np.log(4.0)
    return float(np.clip(sim, 0.0, 1.0))

def rrf_fuse(ranked_lists, k=60):
    scores = {}
    for lst in ranked_lists:
        for rank, cid in enumerate(lst):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
    return scores

def ppm_window_indices(sorted_masses, query_mass, ppm):
    tol = query_mass * ppm * 1e-6
    lo = np.searchsorted(sorted_masses, query_mass - tol, side="left")
    hi = np.searchsorted(sorted_masses, query_mass + tol, side="right")
    return lo, hi
print("core library loaded")


In [ ]:

# ================= Load data =================
print("=== /kaggle/input contents ===")
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    if level > 3:
        continue
    print("DIR:", root)
    for f in files[:20]:
        print("  -", f)

cands = glob.glob("/kaggle/input/**/train.parquet", recursive=True)
print("train.parquet candidates:", cands)
assert cands, "train.parquet not found under /kaggle/input -- attach the competition data via + Add Input"
INPUT_DIR = os.path.dirname(cands[0])
print("input dir:", INPUT_DIR)

t0 = time.time()
train = pq.read_table(os.path.join(INPUT_DIR, "train.parquet")).to_pandas()
test = pq.read_table(os.path.join(INPUT_DIR, "test.parquet")).to_pandas()
print(f"loaded in {time.time()-t0:.1f}s")
print("train:", train.shape, " test:", test.shape)
print("train cols:", list(train.columns))
print("test cols:", list(test.columns))
print("train adducts:", sorted(train["adduct"].dropna().unique()))
print("test adducts:", sorted(test["adduct"].dropna().unique()))
print("ingest_lib values:", sorted(train["ingest_lib"].dropna().unique()))


In [ ]:

# ================= Preprocess train + build neutral-mass index =================
RUN_VALIDATION = True   # set False to skip validation and go straight to prediction

t0 = time.time()
n = len(train)
tr_mz = train["ms2_mzs"].to_numpy()
tr_int = train["ms2_normalized_intensities"].to_numpy()
tr_prec = train["precursor_mz"].to_numpy(dtype=np.float64)
tr_adduct = train["adduct"].to_numpy()
tr_key14 = train["inchikey14"].to_numpy()
tr_smiles = train["normalized_smiles"].to_numpy()
np_mask = (train["ingest_lib"] == "enveda-np-examples").to_numpy()
print(f"np-examples spectra: {np_mask.sum()} / {n}")

tr_neutral = neutral_mass(tr_prec, tr_adduct)

# preprocess each spectrum
lib_mzs, lib_int = [], []
for i in range(n):
    pm, pi = preprocess_peaks(tr_mz[i], tr_int[i], tr_prec[i])
    lib_mzs.append(pm); lib_int.append(pi)
    if (i + 1) % 500000 == 0:
        print(f"  preprocessed {i+1}/{n} ({time.time()-t0:.0f}s)")
lib_mzs = np.array(lib_mzs, dtype=object)
lib_int = np.array(lib_int, dtype=object)

# free the heavy raw columns (keep only what validation/predict need)
del train, tr_mz, tr_int
import gc; gc.collect()

order = np.argsort(tr_neutral)
lib_neutral_sorted = tr_neutral[order]
lib_pos2row = order
del tr_neutral, order; gc.collect()

key14_to_smiles = {}
for k, s in zip(tr_key14, tr_smiles):
    if k not in key14_to_smiles:
        key14_to_smiles[k] = s
print(f"index built in {time.time()-t0:.1f}s; {len(key14_to_smiles)} unique InChIKey14")


In [ ]:

# ================= Search primitives =================
PPM_WINDOW = 10.0
TOPK_PER_SPECTRUM = 200

def search_one_spectrum(q_mzs, q_int, q_neutral, exclude_rows=None):
    """Return list of (inchikey14, best_sim) sorted desc, top 200."""
    lo, hi = ppm_window_indices(lib_neutral_sorted, q_neutral, PPM_WINDOW)
    best = {}
    for pos in range(lo, hi):
        row = lib_pos2row[pos]
        if exclude_rows is not None and row in exclude_rows:
            continue
        s = entropy_similarity(q_mzs, q_int, lib_mzs[row], lib_int[row])
        if s <= 0:
            continue
        k = tr_key14[row]
        if s > best.get(k, 0.0):
            best[k] = s
    ranked = sorted(best.items(), key=lambda kv: -kv[1])[:TOPK_PER_SPECTRUM]
    return [k for k, _ in ranked]

def predict_molecule(spectra, exclude_rows=None):
    """
    spectra: list of (mzs, intensities, precursor_mz, adduct).
    Returns ranked list of inchikey14 (RRF over per-spectrum lists).
    """
    per_spec = []
    neutrals = []
    for mzs, inten, prec, ad in spectra:
        qm, qi = preprocess_peaks(mzs, inten, prec)
        qn = neutral_mass(prec, ad)
        neutrals.append(qn)
        if len(qm) == 0:
            continue
        per_spec.append(search_one_spectrum(qm, qi, qn, exclude_rows))
    if not per_spec:
        # fallback: nearest-mass structures
        qn = float(np.median(neutrals)) if neutrals else 0.0
        lo, hi = ppm_window_indices(lib_neutral_sorted, qn, 100.0)
        cands = [tr_key14[lib_pos2row[p]] for p in range(lo, min(hi, lo + 25))]
        seen, out = set(), []
        for c in cands:
            if c not in seen:
                seen.add(c); out.append(c)
        return out
    scores = rrf_fuse(per_spec, k=60)
    return sorted(scores, key=lambda k: -scores[k])

def fill_to_25(ranked, q_neutral):
    """Dedupe (already by key14) and fill to 25 with nearest-mass structures.
    Expanding ppm window guarantees 25 whenever the library has >=25 structures."""
    out, seen = [], set()
    for k in ranked:
        if k not in seen:
            seen.add(k); out.append(k)
        if len(out) >= 25:
            return out
    for ppm in (500.0, 5000.0, 50000.0):
        lo, hi = ppm_window_indices(lib_neutral_sorted, q_neutral, ppm)
        # walk outward from closest mass: alternate below/above insertion point
        mid = (lo + hi) // 2
        a, b = mid - 1, mid
        while (a >= lo or b < hi) and len(out) < 25:
            if a >= lo:
                k = tr_key14[lib_pos2row[a]]
                if k not in seen:
                    seen.add(k); out.append(k)
                a -= 1
            if b < hi and len(out) < 25:
                k = tr_key14[lib_pos2row[b]]
                if k not in seen:
                    seen.add(k); out.append(k)
                b += 1
        if len(out) >= 25:
            break
    if len(out) < 25:
        # final guarantee: sweep library in mass order
        for p in range(len(lib_neutral_sorted)):
            k = tr_key14[lib_pos2row[p]]
            if k not in seen:
                seen.add(k); out.append(k)
            if len(out) >= 25:
                break
    return out
print("search functions ready")


In [ ]:

# ================= Validation: hold out enveda-np-examples =================
if RUN_VALIDATION:
    t0 = time.time()
    np_rows = set(np.nonzero(np_mask)[0].tolist())
    print(f"holding out {len(np_rows)} np-examples spectra")

    np_by_mol = defaultdict(list)
    for row in np_rows:
        np_by_mol[tr_key14[row]].append(row)
    print(f"{len(np_by_mol)} held-out molecules")

    rr_sum, n_mol = 0.0, 0
    for true_k, row_list in np_by_mol.items():
        per_spec, neutrals = [], []
        for r in row_list:
            qm, qi = lib_mzs[r], lib_int[r]
            qn = float(neutral_mass(float(tr_prec[r]), str(tr_adduct[r])))
            neutrals.append(qn)
            if len(qm) == 0:
                continue
            per_spec.append(search_one_spectrum(qm, qi, qn, np_rows))
        ranked = []
        if per_spec:
            scores = rrf_fuse(per_spec, k=60)
            ranked = sorted(scores, key=lambda k: -scores[k])
        qn = float(np.median(neutrals)) if neutrals else 0.0
        ranked = fill_to_25(ranked, qn)
        try:
            rr_sum += 1.0 / (ranked.index(true_k) + 1)
        except ValueError:
            pass
        n_mol += 1
        if n_mol % 50 == 0:
            print(f"  {n_mol}/{len(np_by_mol)}  running MRR={rr_sum/n_mol:.4f}")
    print(f"VALIDATION MRR@25 (np-examples holdout, Class-1 proxy): {rr_sum/n_mol:.4f}  ({time.time()-t0:.0f}s)")
else:
    print("validation skipped")


In [ ]:

# ================= Predict hidden test set =================
t0 = time.time()
te_mz = test["ms2_mzs"].to_numpy()
te_int = test["ms2_normalized_intensities"].to_numpy()
te_prec = test["precursor_mz"].to_numpy(dtype=np.float64)
te_adduct = test["adduct"].to_numpy()
te_mol = test["molecule_id"].to_numpy()

mol_rows = defaultdict(list)
for i, m in enumerate(te_mol):
    mol_rows[m].append(i)
print(f"{len(mol_rows)} test molecules, {len(test)} spectra")

rows = []
for j, (mol, idxs) in enumerate(mol_rows.items()):
    spectra = [(
        np.asarray(te_mz[i], dtype=np.float64),
        np.asarray(te_int[i], dtype=np.float64),
        float(te_prec[i]), str(te_adduct[i]),
    ) for i in idxs]
    ranked = predict_molecule(spectra)
    qn = float(np.median([neutral_mass(p, a) for _, _, p, a in spectra]))
    ranked = fill_to_25(ranked, qn)
    smiles = [key14_to_smiles[k] for k in ranked[:25]]
    rows.append((mol, ";".join(smiles)))
    if (j + 1) % 100 == 0:
        print(f"  {j+1}/{len(mol_rows)} ({time.time()-t0:.0f}s)")

sub = pd.DataFrame(rows, columns=["molecule_id", "smiles"])
assert sub["molecule_id"].is_unique and len(sub) == len(mol_rows)
assert (sub["smiles"].str.count(";") <= 24).all()
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"submission written: {sub.shape} in {time.time()-t0:.0f}s")
print(sub.head(3).to_string())
